# Navigation

Map the track by driving around (**SLAM**), then let the robot drive itself to any point you click (**Nav2**). This is the standard ROS 2 navigation stack — slam_toolbox for mapping, AMCL + Nav2 (MPPI controller, Hybrid-A* planner) for autonomous driving — configured for PhysiCar's Ackermann steering in `assets/navigation/config/`. The runnable cells are shell cells (`%%bash`).

## [1] Mapping (SLAM)

### 1.1. Start mapping

The map origin is wherever mapping starts, so the cell first puts the robot at the start line (SIM: automatic respawn; real robot: place it there yourself). It launches RViz and slam_toolbox in the background and returns — then **drive the robot around** with the App's keyboard control and watch the map grow in the **VNC tab**. Drive slowly and cover the whole track; loop closures (revisiting a place) sharpen the map.

In [ ]:
%%bash
# Start mapping: drive the robot around and watch the map grow in the VNC
# tab. When you are done, run [2] — it saves the map automatically.
DIR=$(pwd)/assets/navigation
[ -d "$DIR" ] || DIR=$(pwd)/examples/assets/navigation   # cwd = workspace root
bg() { setsid "$@" > /dev/null 2>&1 < /dev/null & }      # detach from this cell

pkill -f "[s]lam_toolbox|[n]av2|[c]omponent_container|[r]viz2" && sleep 3

# The map origin is wherever mapping starts. SIM: reset puts the car (and every
# object) back at its start pose, so the origin is always the start line. Real robot: place it there.
grep -q "SIM=true" /opt/physicar/userdata/.env 2>/dev/null && SIM=true || SIM=false
if [ $SIM = true ]; then
    curl -s -X POST http://localhost/sim/api/reset > /dev/null   # applied on return
fi

bg rviz2 -d $DIR/config/slam.rviz --ros-args -p use_sim_time:=$SIM
bg ros2 launch slam_toolbox online_async_launch.py \
    slam_params_file:=$DIR/config/slam_params.yaml use_sim_time:=$SIM

echo "mapping — drive the robot, then run [2]"

## [2] Autonomous driving (Nav2)

### 2.1. Save the map & start Nav2

Saves the map from the running SLAM (`map/my_map.png` + `.yaml`), swaps SLAM out for AMCL localization, brings up the Nav2 stack (~30 s), and opens the navigation RViz view. The robot starts localized at the start line.

Then, in RViz (VNC tab):

- **[2D Pose Estimate]** — click-drag where the robot *really* is, if the initial guess is off
- **[2D Goal Pose]** — click a destination and drag toward the goal heading — off it goes

In [ ]:
%%bash
# Autonomous driving (saves the map from the running SLAM automatically).
DIR=$(pwd)/assets/navigation
[ -d "$DIR" ] || DIR=$(pwd)/examples/assets/navigation   # cwd = workspace root
bg() { setsid "$@" > /dev/null 2>&1 < /dev/null & }      # detach from this cell

grep -q "SIM=true" /opt/physicar/userdata/.env 2>/dev/null && SIM=true || SIM=false

# SLAM is running — save its map now (open map/my_map.png to check it)
if pgrep -f "[a]sync_slam_toolbox" > /dev/null; then
    mkdir -p $DIR/map
    ros2 run nav2_map_server map_saver_cli -f $DIR/map/my_map --fmt png
fi

# SLAM also publishes map->odom — swap it (and any previous run) out
pkill -f "[s]lam_toolbox|[n]av2|[c]omponent_container|[r]viz2" && sleep 3
[ -f $DIR/map/my_map.yaml ] || { echo "no map — run [1] first"; exit 1; }

# AMCL starts localized at the map origin = the start line (nav2_params
# set_initial_pose). SIM: reset there. Real robot: place it there.
if [ $SIM = true ]; then
    curl -s -X POST http://localhost/sim/api/reset > /dev/null   # applied on return
fi

bg ros2 launch nav2_bringup localization_launch.py \
    map:=$DIR/map/my_map.yaml params_file:=$DIR/config/nav2_params.yaml \
    use_sim_time:=$SIM
bg ros2 launch nav2_bringup navigation_launch.py \
    params_file:=$DIR/config/nav2_params.yaml use_sim_time:=$SIM

echo "starting nav2 (about 30s) ..."
until timeout 2 ros2 lifecycle get /amcl 2>/dev/null | grep -q active; do sleep 2; done
until timeout 2 ros2 lifecycle get /controller_server 2>/dev/null | grep -q active; do sleep 2; done

bg rviz2 -d $DIR/config/nav.rviz --ros-args -p use_sim_time:=$SIM
sleep 5   # rviz sometimes misses the latched map — republish once it is up
ros2 service call /map_server/load_map nav2_msgs/srv/LoadMap \
    "{map_url: $DIR/map/my_map.yaml}" > /dev/null

echo "ready — click a goal in RViz (VNC tab)"

### 2.2. The saved map

In [ ]:
import os

from IPython.display import Image, display

DIR = "assets/navigation"
if not os.path.isdir(DIR):               # kernel cwd is the workspace root
    DIR = "examples/" + DIR

p = f"{DIR}/map/my_map.png"
if os.path.exists(p):
    print(open(f"{DIR}/map/my_map.yaml").read())
    display(Image(filename=p, width=300))
else:
    print("no map yet — run [1], drive around, then run [2]")

## [3] Stop & tune

### 3.1. Stop everything

In [ ]:
%%bash
pkill -f "[s]lam_toolbox|[n]av2|[c]omponent_container|[r]viz2" || true

### 3.2. Tuning

- `assets/navigation/config/slam_params.yaml` — mapping: resolution 0.05 m, `max_laser_range` 16 m, loop-closure knobs
- `assets/navigation/config/nav2_params.yaml` — driving: costmap inflation, MPPI controller weights, planner settings, AMCL

Edit, then re-run [1]/[2] — the cells kill previous runs on start.